# T-SQL Programming — Variables, Functions and Stored Procedures

This notebook extends **DemoDB** built in notebook 01. By the end you will have added:

- Three tables: `Sales.Product`, `Sales.CustomerOrder`, `Sales.CustomerOrderLine`
- Two scalar functions: `Sales.fn_LineTotal`, `Sales.fn_FullName`
- One inline table-valued function: `Sales.fn_GetCustomerOrders`
- Two stored procedures: `Sales.usp_SearchCustomers`, `Sales.usp_AddOrder`

### How the object hierarchy grows

```
DemoDB
└── Schema: Sales
    ├── Table:     Sales.Customer               ← notebook 01
    ├── View:      Sales.vw_CustomerSummary     ← notebook 01
    ├── Table:     Sales.Product                ← NEW
    ├── Table:     Sales.CustomerOrder          ← NEW
    ├── Table:     Sales.CustomerOrderLine      ← NEW
    ├── Function:  Sales.fn_LineTotal           ← NEW  (scalar)
    ├── Function:  Sales.fn_FullName            ← NEW  (scalar)
    ├── Function:  Sales.fn_GetCustomerOrders   ← NEW  (inline TVF)
    ├── Procedure: Sales.usp_SearchCustomers    ← NEW
    └── Procedure: Sales.usp_AddOrder           ← NEW
```

### Prerequisites

- **Notebook 01** completed with all cells run: `DemoDB` exists, `Sales.Customer` has 5 rows
- VS Code connected to `localhost` via the mssql extension

### How to run cells

Click the **▶ Run Cell** button or press `Ctrl+Shift+E`.
Run cells **in order from top to bottom** — each section builds on the previous one.

> ⚠️ **Before running any cell** confirm your mssql connection targets `localhost`.
> Use `Ctrl+Shift+P` → **MS SQL: Connect** if needed.

---

## Prerequisites Check

The cell below verifies that `DemoDB` is reachable and `Sales.Customer` contains the
5 seed rows inserted in notebook 01. If the cell errors or returns fewer than 5 rows,
run notebook 01 (Sections 1–6) before continuing here.

In [ ]:
-- ─── Prerequisites check ────────────────────────────────────────────────────
-- This notebook chains from 01_database_fundamentals.ipynb.
-- Expected result: CustomerCount = 5, MinID = 1, MaxID = 5
USE DemoDB;
GO

SELECT
    COUNT(*)         AS CustomerCount,
    MIN(CustomerID)  AS MinID,
    MAX(CustomerID)  AS MaxID
FROM [Sales].[Customer];

---

## Section 1 — Expand the Schema

Before exploring T-SQL programming constructs we need richer data to work with.
This section adds three related tables to the `Sales` schema:

```
Sales.Customer          ← already exists (notebook 01)
    ↑ 1:many
Sales.CustomerOrder     ← one row per customer transaction
    ↑ 1:many
Sales.CustomerOrderLine ← one row per product line within an order
    ↑ many:1
Sales.Product           ← product catalogue
```

**Why `CustomerOrder` and not `Order`?**
`ORDER` is a reserved keyword in T-SQL (used in `ORDER BY`). Using it as an unbracketed
table name causes confusing parse errors. `CustomerOrder` avoids the problem entirely —
no brackets required in daily use.

> **↕ Snowflake vs SQL Server — Foreign key enforcement**
>
> In SQL Server, foreign keys are **defined and enforced** by default. An `INSERT` that
> references a non-existent parent row is immediately rejected with a constraint error.
> In Snowflake, foreign keys are **defined but not enforced** — they exist as metadata
> for documentation and tooling only; an orphaned child row inserts without complaint.
>
> This is one of the most important behavioural differences between the two platforms.
> FK constraints will be explored in depth in notebook 04.

### Naming conventions

Consistent prefixes make object types recognisable at a glance — in queries, Object
Explorer, and error messages. This series uses:

| Object type | Prefix | Example |
|-------------|--------|---------|
| Stored procedure | `usp_` | `Sales.usp_AddOrder` |
| Scalar function | `fn_` | `Sales.fn_LineTotal` |
| Inline TVF | `fn_` | `Sales.fn_GetCustomerOrders` |
| View | `vw_` | `Sales.vw_CustomerSummary` (notebook 01) |

In [ ]:
-- ─── Section 1a: Create tables ──────────────────────────────────────────────
USE DemoDB;
GO

-- Drop in child-first order: FK constraints prevent dropping a parent while a
-- child table still references it.
DROP TABLE IF EXISTS [Sales].[CustomerOrderLine];
DROP TABLE IF EXISTS [Sales].[CustomerOrder];
DROP TABLE IF EXISTS [Sales].[Product];
GO

-- Product catalogue
CREATE TABLE [Sales].[Product]
(
    ProductID    INT            IDENTITY(1,1)  NOT NULL,
    ProductName  NVARCHAR(100)                 NOT NULL,
    Category     NVARCHAR(50)                  NOT NULL,
    UnitPrice    MONEY                         NOT NULL,
    CONSTRAINT PK_Product PRIMARY KEY (ProductID)
);
GO

-- One row per customer transaction
CREATE TABLE [Sales].[CustomerOrder]
(
    OrderID     INT           IDENTITY(1,1)       NOT NULL,
    CustomerID  INT                               NOT NULL,
    OrderDate   DATE          DEFAULT GETDATE()   NOT NULL,
    Status      NVARCHAR(20)  DEFAULT N'Pending'  NOT NULL,
    CONSTRAINT PK_CustomerOrder  PRIMARY KEY (OrderID),
    CONSTRAINT FK_Order_Customer FOREIGN KEY (CustomerID)
        REFERENCES [Sales].[Customer] (CustomerID)
);
GO

-- One row per product line within an order
CREATE TABLE [Sales].[CustomerOrderLine]
(
    OrderLineID  INT    IDENTITY(1,1)  NOT NULL,
    OrderID      INT                   NOT NULL,
    ProductID    INT                   NOT NULL,
    Quantity     INT                   NOT NULL,
    UnitPrice    MONEY                 NOT NULL,   -- price captured at order time
    CONSTRAINT PK_CustomerOrderLine  PRIMARY KEY (OrderLineID),
    CONSTRAINT FK_OrderLine_Order    FOREIGN KEY (OrderID)
        REFERENCES [Sales].[CustomerOrder] (OrderID),
    CONSTRAINT FK_OrderLine_Product  FOREIGN KEY (ProductID)
        REFERENCES [Sales].[Product] (ProductID)
);
GO

-- Verify all five Sales tables now exist
SELECT TABLE_NAME, TABLE_TYPE
FROM   INFORMATION_SCHEMA.TABLES
WHERE  TABLE_SCHEMA = 'Sales'
ORDER BY TABLE_NAME;

In [ ]:
-- ─── Section 1b: Seed data ──────────────────────────────────────────────────
USE DemoDB;
GO

-- 5 products
INSERT INTO [Sales].[Product] (ProductName, Category, UnitPrice)
VALUES
    (N'Laptop Pro 15',       N'Electronics',  1299.99),
    (N'Wireless Mouse',      N'Electronics',    29.99),
    (N'Mechanical Keyboard', N'Electronics',    79.99),
    (N'27" Monitor',         N'Electronics',   399.99),
    (N'USB-C Hub',           N'Accessories',    49.99);
GO

-- 3 orders (CustomerIDs 1 = Alice, 2 = Ben from notebook 01)
INSERT INTO [Sales].[CustomerOrder] (CustomerID, OrderDate, Status)
VALUES
    (1, '2024-02-10', N'Completed'),   -- Alice, first order
    (2, '2024-03-18', N'Completed'),   -- Ben
    (1, '2024-04-25', N'Pending');     -- Alice, second order
GO

-- 8 order lines across the 3 orders
INSERT INTO [Sales].[CustomerOrderLine] (OrderID, ProductID, Quantity, UnitPrice)
VALUES
    -- Order 1 (Alice): Laptop, Mouse, USB-C Hub
    (1, 1, 1, 1299.99),
    (1, 2, 1,   29.99),
    (1, 5, 1,   49.99),
    -- Order 2 (Ben): Keyboard + Monitor
    (2, 3, 1,  79.99),
    (2, 4, 1, 399.99),
    -- Order 3 (Alice): 2× Mouse, Keyboard, USB-C Hub
    (3, 2, 2,  29.99),
    (3, 3, 1,  79.99),
    (3, 5, 1,  49.99);
GO

-- Verify row counts
SELECT 'Product'           AS TableName, COUNT(*) AS Rows FROM [Sales].[Product]
UNION ALL
SELECT 'CustomerOrder',                  COUNT(*)         FROM [Sales].[CustomerOrder]
UNION ALL
SELECT 'CustomerOrderLine',              COUNT(*)         FROM [Sales].[CustomerOrderLine];

### What just happened?

Three related tables have been added to the `Sales` schema with FK constraints wiring them together.

**`UnitPrice` in `CustomerOrderLine`** captures the product price *at the time of the order*,
independent of the product's current price. If `Sales.Product` is updated later, historical
order totals stay accurate. This is a standard denormalisation pattern in order systems — the
order line owns its own price rather than inheriting it dynamically from the product.

The FK constraints enforce parent-child integrity at the database level immediately.
Try inserting an order line with a non-existent `OrderID` and it will fail — the database
is the last line of defence, even when application code has a bug.

---

## Section 2 — T-SQL Variables

A **variable** holds a typed value for the duration of a T-SQL batch. Variables are the
foundation of all T-SQL programming — used inside functions, stored procedures, and
standalone scripts to store intermediate results and drive conditional logic.

Key rules:
- Declared with `DECLARE` before first use.
- Names always start with `@` (e.g. `@CustomerID`, `@TotalAmount`).
- Scoped to a single **batch** — they do not survive a `GO` separator.

> **↕ Snowflake vs SQL Server — Variables**
>
> Snowflake Scripting (inside a `BEGIN...END` block) uses `LET x := value;` — no `@` prefix.
> T-SQL uses `DECLARE @x TYPE; SET @x = value;`. The `@` prefix is mandatory in T-SQL and
> makes variables visually distinct from column names inside complex queries — a deliberate
> design choice that reduces ambiguity.

### Syntax

```sql
DECLARE @VarName DataType [= initial_value];  -- declare (and optionally initialise)

SET @VarName = expression;                    -- assign a literal or computed value

SELECT @VarName = ColumnName                  -- assign from a query result
FROM   SomeTable
WHERE  condition;

SELECT * FROM SomeTable WHERE ID = @VarName;  -- use in a query
```

**`SET` vs `SELECT` for assignment:**
- Use `SET` to assign a literal or computed expression.
- Use `SELECT @var = col FROM table` to read a value from a table.
  If the query returns multiple rows, `@var` gets the **last** value evaluated.
  Use `TOP 1 ... ORDER BY ...` to make the result deterministic.

**Batch scope** is demonstrated at the end of the cell below — a variable declared
before `GO` is inaccessible after it.

In [ ]:
-- ─── Section 2: T-SQL Variables ─────────────────────────────────────────────
USE DemoDB;
GO

-- ── 2a: Declare and assign a literal value ───────────────────────────────────
DECLARE @CustomerID  INT          = 1;
DECLARE @Label       NVARCHAR(50) = N'Target customer ID:';

-- PRINT sends output to the Messages tab (not the grid)
PRINT @Label;
SELECT @CustomerID AS TargetCustomerID;


-- ── 2b: Assign from a table query ────────────────────────────────────────────
-- @CustomerID is still in scope — same batch, no GO between 2a and 2b
DECLARE @FullName NVARCHAR(101);

SELECT TOP 1 @FullName = FirstName + N' ' + LastName
FROM   [Sales].[Customer]
WHERE  CustomerID = @CustomerID;   -- reusing @CustomerID from 2a

SELECT @FullName AS CustomerFullName;


-- ── 2c: Use a variable in a WHERE clause ─────────────────────────────────────
DECLARE @StatusFilter NVARCHAR(20) = N'Completed';

SELECT
    co.OrderID,
    co.OrderDate,
    co.Status,
    c.FirstName + N' ' + c.LastName  AS CustomerName
FROM [Sales].[CustomerOrder] co
JOIN [Sales].[Customer]       c  ON c.CustomerID = co.CustomerID
WHERE co.Status = @StatusFilter;


-- ── 2d: Batch scope — @StatusFilter is still alive here ──────────────────────
SELECT @StatusFilter AS StillAliveBeforeGO;
GO

-- After GO, all variables declared above are gone.
-- Uncommenting the line below would raise:
--   Msg 137: Must declare the scalar variable "@StatusFilter"
-- SELECT @StatusFilter;

### What just happened?

- **2a** declared two variables and printed one to the Messages tab. `PRINT` is a useful
  debugging tool inside scripts and stored procedures — its output appears in the Messages
  pane, separate from query result grids.
- **2b** read a name from `Sales.Customer` into `@FullName` using `SELECT @var = col`.
  It also reused `@CustomerID` from 2a — both are alive in the same batch.
- **2c** filtered orders using `@StatusFilter`, showing how variables turn hardcoded
  literals into named, self-documenting values.
- **2d** confirmed `@StatusFilter` is still accessible before the `GO`, then illustrated
  that it disappears after the batch separator.

This batch-scope behaviour is why stored procedures work the way they do: each call to a
procedure is its own execution context, with fresh variables every time.

---

## Section 3 — Scalar Functions

A **scalar function** takes zero or more input parameters and returns exactly one value.
It is called inline — anywhere an expression is valid: inside `SELECT`, `WHERE`, `SET`, or
even another function.

Scalar functions encapsulate reusable calculations and string operations, keeping query
logic DRY and maintainable.

**Performance caveat — important for BI work:**
T-SQL scalar functions execute **row-by-row** and cannot be parallelised by the query
optimizer. On a table with millions of rows this overhead compounds quickly. For simple
transformations on large result sets, prefer an inline expression in the query or an
inline table-valued function (Section 4). Scalar functions are well suited for
occasional use, formatting, and logic called on small datasets.

> **↕ Snowflake vs SQL Server — Scalar UDFs**
>
> Snowflake scalar UDFs support SQL, JavaScript, Python, and Java as the function body —
> non-SQL implementations can be vectorised and avoid the row-by-row overhead.
> SQL Server T-SQL scalar UDFs are SQL-only and always row-by-row.
> The `CREATE FUNCTION` / `RETURNS` syntax is nearly identical between the two platforms.

### Syntax

```sql
DROP FUNCTION IF EXISTS [Schema].[fn_Name];
GO

CREATE FUNCTION [Schema].[fn_Name]
(
    @Param1 DataType,
    @Param2 DataType
)
RETURNS ReturnType
AS
BEGIN
    DECLARE @Result ReturnType;
    -- ... logic ...
    RETURN @Result;
END;
GO
```

- `CREATE FUNCTION` must be the **first statement in a batch** — always place `GO` before it.
- Functions are **read-only** — no `INSERT`, `UPDATE`, or `DELETE` inside a scalar function.
  Use stored procedures for write operations.
- Call syntax: `Schema.fn_Name(arg1, arg2)` — no `EXEC`, just an inline expression.

In [ ]:
-- ─── Section 3: Scalar Functions ────────────────────────────────────────────
USE DemoDB;
GO

-- ── fn_LineTotal: quantity × unit price ─────────────────────────────────────
DROP FUNCTION IF EXISTS [Sales].[fn_LineTotal];
GO

CREATE FUNCTION [Sales].[fn_LineTotal]
(
    @Quantity   INT,
    @UnitPrice  MONEY
)
RETURNS MONEY
AS
BEGIN
    RETURN @Quantity * @UnitPrice;
END;
GO


-- ── fn_FullName: concatenate first and last name ─────────────────────────────
DROP FUNCTION IF EXISTS [Sales].[fn_FullName];
GO

CREATE FUNCTION [Sales].[fn_FullName]
(
    @FirstName  NVARCHAR(50),
    @LastName   NVARCHAR(50)
)
RETURNS NVARCHAR(101)
AS
BEGIN
    RETURN @FirstName + N' ' + @LastName;
END;
GO


-- ── Demo: call both functions in a SELECT across the joined tables ────────────
SELECT
    col.OrderLineID,
    col.OrderID,
    p.ProductName,
    col.Quantity,
    col.UnitPrice,
    [Sales].[fn_LineTotal](col.Quantity, col.UnitPrice)   AS LineTotal,
    [Sales].[fn_FullName](c.FirstName, c.LastName)        AS OrderedBy
FROM [Sales].[CustomerOrderLine]  col
JOIN [Sales].[CustomerOrder]       co  ON co.OrderID   = col.OrderID
JOIN [Sales].[Product]             p   ON p.ProductID  = col.ProductID
JOIN [Sales].[Customer]            c   ON c.CustomerID = co.CustomerID
ORDER BY col.OrderLineID;

### What just happened?

- **`fn_LineTotal`** wraps a single multiplication. It may seem trivial, but naming the
  calculation makes every query that uses it self-documenting — and if the pricing formula
  ever changes (e.g. adding a discount multiplier), you edit one function rather than
  hunting down every query.

- **`fn_FullName`** encapsulates the `FirstName + ' ' + LastName` pattern already used in
  `Sales.vw_CustomerSummary` (notebook 01). You could now update that view to call
  `[Sales].[fn_FullName](FirstName, LastName)` — one definition, used everywhere.

The schema-qualified call syntax `[Sales].[fn_LineTotal](...)` is mandatory for user-defined
scalar functions in SQL Server. Omitting the schema prefix raises an error — unlike built-in
functions such as `GETDATE()` or `LEN()`, which need no qualifier.

---

## Section 4 — Inline Table-Valued Functions

An **inline table-valued function (TVF)** returns a `TABLE` instead of a single value.
It contains exactly one `SELECT` statement and is evaluated **inline** by the query
optimizer — the engine treats it as if you had written the `SELECT` directly, which
avoids the row-by-row penalty of scalar UDFs.

Think of an inline TVF as a **parameterised view**: all the benefits of a view, plus
the ability to accept arguments.

**Multi-statement TVFs** (which use `BEGIN...END` and accumulate results into a table
variable) are a legacy pattern with poor optimizer integration. You may encounter them
in older databases, but avoid writing new ones — prefer inline TVFs instead.

> **↕ Snowflake vs SQL Server — Table-valued functions**
>
> Snowflake has **UDTFs** (User-Defined Table Functions) — the concept is identical:
> a function that returns a table. Snowflake UDTFs can be written in JavaScript or Python,
> enabling more complex logic than a single SQL statement. SQL Server inline TVFs are
> SQL-only but integrate seamlessly with the query optimizer.

### Syntax

```sql
CREATE FUNCTION [Schema].[fn_Name]
(
    @Param DataType
)
RETURNS TABLE
AS
RETURN
(
    -- Single SELECT statement — no BEGIN...END
    SELECT col1, col2, ...
    FROM   SomeTable
    WHERE  condition = @Param
);
GO
```

- `RETURNS TABLE` — no column list needed; the schema is inferred from the `SELECT`.
- `AS RETURN (...)` — no `BEGIN...END`, just the query in parentheses.
- **Called with `FROM` or `CROSS APPLY`:**

```sql
-- Direct call (single input value)
SELECT * FROM [Sales].[fn_GetCustomerOrders](1);

-- CROSS APPLY (one call per row of the outer table — like a correlated join)
SELECT c.FirstName, o.ProductName
FROM   [Sales].[Customer]                                  c
CROSS APPLY [Sales].[fn_GetCustomerOrders](c.CustomerID)   o;
```

`CROSS APPLY` excludes outer rows with no matching function results (like `INNER JOIN`).
Use `OUTER APPLY` to keep all outer rows, filling function columns with `NULL` when
there are no matches (like `LEFT JOIN`).

In [ ]:
-- ─── Section 4: Inline Table-Valued Function ─────────────────────────────────
USE DemoDB;
GO

DROP FUNCTION IF EXISTS [Sales].[fn_GetCustomerOrders];
GO

CREATE FUNCTION [Sales].[fn_GetCustomerOrders]
(
    @CustomerID INT
)
RETURNS TABLE
AS
RETURN
(
    SELECT
        co.OrderID,
        co.OrderDate,
        co.Status,
        p.ProductName,
        p.Category,
        col.Quantity,
        col.UnitPrice,
        [Sales].[fn_LineTotal](col.Quantity, col.UnitPrice)  AS LineTotal
    FROM [Sales].[CustomerOrder]     co
    JOIN [Sales].[CustomerOrderLine] col ON col.OrderID  = co.OrderID
    JOIN [Sales].[Product]           p   ON p.ProductID  = col.ProductID
    WHERE co.CustomerID = @CustomerID
);
GO


-- ── Demo 1: Direct call for a single customer ────────────────────────────────
SELECT * FROM [Sales].[fn_GetCustomerOrders](1);   -- Alice's orders
GO


-- ── Demo 2: CROSS APPLY — all customers with at least one order ───────────────
SELECT
    c.CustomerID,
    [Sales].[fn_FullName](c.FirstName, c.LastName)       AS CustomerName,
    o.OrderID,
    o.OrderDate,
    o.ProductName,
    o.Quantity,
    o.LineTotal
FROM [Sales].[Customer]                                  c
CROSS APPLY [Sales].[fn_GetCustomerOrders](c.CustomerID) o
ORDER BY c.CustomerID, o.OrderID;

### What just happened?

`fn_GetCustomerOrders` joins four tables and calls `fn_LineTotal` — all hidden behind a
single parameterised interface. The query optimizer sees through the TVF and builds a single
execution plan, just as if those joins were written inline.

**Demo 1** called the function directly for one customer — same syntax as querying any table.

**Demo 2** used `CROSS APPLY` to call the function once per customer row. Customers with no
orders (Clara, David, Emma) are excluded because `CROSS APPLY` acts like an `INNER JOIN`.
Notice that `fn_FullName` from Section 3 composes naturally here — functions build on
each other, keeping the outer query clean and readable.

**Why this matters for BI work:** inline TVFs are the SQL Server equivalent of a
parameterised dataset or a dynamic DAX measure — reusable query logic with inputs.
Power BI report authors querying SQL Server can call a TVF from DirectQuery mode just
as they would a table or view.

---

## Section 5 — Stored Procedures (Input Parameters)

A **stored procedure** is a named, compiled batch of T-SQL stored in the database.
Unlike functions, stored procedures:

- **Can modify data** — `INSERT`, `UPDATE`, `DELETE` are all permitted.
- **Cannot be called inside a `SELECT`** — invoked with `EXEC` or `EXECUTE` only.
- **Can produce multiple result sets** — several `SELECT` statements, one after another.
- **Can use transactions and error handling** — covered in notebook 05.

Parameters are declared the same way as function parameters: `@Name Type [= default]`.
A default value makes the parameter optional — the caller may omit it.

> **↕ Snowflake vs SQL Server — Stored Procedures**
>
> Snowflake stored procedures are written in Snowflake Scripting, JavaScript, or Python —
> not T-SQL. The calling syntax differs too:
>
> | Platform | Call syntax |
> |----------|-------------|
> | SQL Server | `EXEC Sales.usp_SearchCustomers @LastName = N'C%'` |
> | Snowflake | `CALL schema.proc_name('arg')` |
>
> The concept — a named routine that encapsulates logic and accepts parameters — is
> identical; only the implementation language and call syntax differ.

### Syntax

```sql
CREATE PROCEDURE [Schema].[usp_Name]
    @Param1 DataType,
    @Param2 DataType = default_value   -- default makes this parameter optional
AS
BEGIN
    SET NOCOUNT ON;   -- suppress "(N rows affected)" messages — always use in SPs
    -- ... T-SQL logic ...
END;
GO

-- Call with named parameters (preferred — order-independent and self-documenting)
EXEC [Schema].[usp_Name] @Param1 = value1, @Param2 = value2;

-- Call with positional parameters (works but fragile — breaks if parameter order changes)
EXEC [Schema].[usp_Name] value1, value2;
```

**`SET NOCOUNT ON`** suppresses the `"(N rows affected)"` confirmation messages that SQL
Server sends after every data-modifying statement. In stored procedures this prevents
application code or ORMs from misinterpreting these messages as additional result sets,
and reduces network traffic. It is considered mandatory best practice in any production SP.

In [ ]:
-- ─── Section 5: Stored Procedure — Input Parameters ─────────────────────────
USE DemoDB;
GO

-- ── usp_SearchCustomers: look up customers by last name pattern ───────────────
DROP PROCEDURE IF EXISTS [Sales].[usp_SearchCustomers];
GO

CREATE PROCEDURE [Sales].[usp_SearchCustomers]
    @LastName  NVARCHAR(50)   -- required: no default value
AS
BEGIN
    SET NOCOUNT ON;

    SELECT
        CustomerID,
        [Sales].[fn_FullName](FirstName, LastName)  AS FullName,
        Email,
        CreatedDate
    FROM [Sales].[Customer]
    WHERE LastName LIKE @LastName
    ORDER BY LastName, FirstName;
END;
GO


-- ── Demo: partial match ───────────────────────────────────────────────────────
EXEC [Sales].[usp_SearchCustomers] @LastName = N'%o%';
GO

-- ── Demo: wildcard returns all customers ─────────────────────────────────────
EXEC [Sales].[usp_SearchCustomers] @LastName = N'%';
GO

-- ── Verify the SP is registered in sys.procedures ────────────────────────────
SELECT name, type_desc, create_date
FROM   sys.procedures
WHERE  name = N'usp_SearchCustomers';

### What just happened?

`Sales.usp_SearchCustomers` wraps a `SELECT` behind a stable, named interface:

- Callers pass a `LIKE` pattern without knowing or caring about the underlying table
  structure. If the team later splits `LastName` into `FirstSurname` and `SecondSurname`,
  only the stored procedure changes — no calling code needs updating.
- The SP reuses `fn_FullName` from Section 3, showing how procedures compose with functions.
- Named parameter calls (`@LastName = N'...'`) are the preferred style: they are
  self-documenting and immune to parameter reordering.

The `sys.procedures` catalog view at the end confirms the SP exists and shows its
creation timestamp — useful for auditing when schema objects were last deployed.

---

## Section 6 — Stored Procedures (OUTPUT Parameters)

An **OUTPUT parameter** lets a stored procedure send a value back to the caller in
addition to (or instead of) a result set. This is the standard pattern for returning
a generated identity value after an `INSERT` — the caller receives the new ID without
a separate `SELECT`.

Two mechanisms work together:

| T-SQL | Purpose |
|-------|---------|
| `@Param TYPE OUTPUT` | Marks the parameter as bidirectional (in and out) |
| `SCOPE_IDENTITY()` | Returns the `IDENTITY` value from the most recent `INSERT` in the current scope |

**Why `SCOPE_IDENTITY()` and not `@@IDENTITY`?**
`@@IDENTITY` returns the last identity inserted in the current session across *all tables*,
including rows inserted by triggers. If a trigger on `CustomerOrder` inserts into another
table, `@@IDENTITY` returns the trigger's ID — not the order's ID. `SCOPE_IDENTITY()` is
scoped to the current statement and is always the safe choice.

> **↕ Snowflake vs SQL Server — Returning generated values**
>
> Snowflake stored procedures return values via a `RETURN` statement (any type, but one
> value only). SQL Server has two return mechanisms:
> - `RETURN int_value` — exits the procedure with an integer status code (used for
>   success/error signalling, not data).
> - `OUTPUT` parameters — any data type, the standard choice for returning generated IDs
>   or computed values back to the caller.

### Syntax

```sql
CREATE PROCEDURE [Schema].[usp_Name]
    @InputParam   DataType,
    @OutputParam  DataType  OUTPUT   -- OUTPUT keyword makes this bidirectional
AS
BEGIN
    SET NOCOUNT ON;
    INSERT INTO SomeTable (...) VALUES (...);
    SET @OutputParam = SCOPE_IDENTITY();   -- capture the generated IDENTITY
END;
GO

-- ── Calling pattern ──────────────────────────────────────────────────────────
DECLARE @NewID INT;

EXEC [Schema].[usp_Name]
    @InputParam  = value,
    @OutputParam = @NewID OUTPUT;   -- OUTPUT keyword required on the call too

SELECT @NewID AS GeneratedID;
```

The `OUTPUT` keyword must appear **both** in the procedure definition and on the `EXEC`
call. Omitting it on the call means the variable is passed by value and the returned
ID is silently discarded — a common mistake.

In [ ]:
-- ─── Section 6: Stored Procedure — OUTPUT Parameter ─────────────────────────
USE DemoDB;
GO

-- ── usp_AddOrder: insert a new order, return the generated OrderID ────────────
DROP PROCEDURE IF EXISTS [Sales].[usp_AddOrder];
GO

CREATE PROCEDURE [Sales].[usp_AddOrder]
    @CustomerID  INT,
    @Status      NVARCHAR(20)  = N'Pending',  -- optional: defaults to Pending
    @NewOrderID  INT           OUTPUT          -- returns the generated identity value
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO [Sales].[CustomerOrder] (CustomerID, Status)
    VALUES (@CustomerID, @Status);

    -- Capture the IDENTITY value generated by the INSERT above
    SET @NewOrderID = SCOPE_IDENTITY();
END;
GO


-- ── Demo 1: Add an order for Ben (CustomerID = 2) with explicit Status ────────
DECLARE @OrderID INT;

EXEC [Sales].[usp_AddOrder]
    @CustomerID = 2,
    @Status     = N'Processing',
    @NewOrderID = @OrderID OUTPUT;

SELECT @OrderID AS NewOrderID;
GO


-- ── Demo 2: Add an order for Clara (CustomerID = 3) — omit optional @Status ──
-- @Status is not passed; the procedure uses its default value 'Pending'
DECLARE @OrderID INT;

EXEC [Sales].[usp_AddOrder]
    @CustomerID = 3,
    @NewOrderID = @OrderID OUTPUT;

SELECT @OrderID AS NewOrderID;
GO


-- ── Verify: all 5 orders (3 seeded + 2 just inserted) ────────────────────────
SELECT
    co.OrderID,
    [Sales].[fn_FullName](c.FirstName, c.LastName)  AS CustomerName,
    co.Status,
    co.OrderDate
FROM [Sales].[CustomerOrder] co
JOIN [Sales].[Customer]       c  ON c.CustomerID = co.CustomerID
ORDER BY co.OrderID;

### What just happened?

`Sales.usp_AddOrder` inserts a row into `Sales.CustomerOrder` and returns the generated
`OrderID` to the caller via the `@NewOrderID OUTPUT` parameter.

Key points from the demo:

- **Demo 1** passed all three parameters including `@Status = N'Processing'`. The returned
  `@OrderID` is immediately usable — the application can store it, pass it to another
  procedure, or insert order lines against it.
- **Demo 2** omitted `@Status`, letting the procedure default to `N'Pending'`. This mirrors
  default argument values in Python (`def fn(x, y='default'):`) — the same concept, the same
  benefit: fewer required arguments for the common case.
- The final `SELECT` confirms all 5 orders exist: 3 seeded in Section 1, plus the two added here.

**`SCOPE_IDENTITY()` vs `@@IDENTITY` — why it matters:**
SQL Server triggers fire inside the same session scope. If adding an order were to trigger
an audit log insert, `@@IDENTITY` would return the audit row's ID, silently breaking the
calling code. `SCOPE_IDENTITY()` always returns the ID from the statement you care about.

---

## Section 7 — Review and Cleanup

### What you built

The complete state of `DemoDB` after both notebooks:

```
SQL Server Instance (localhost)
└── Login:    DemoUser                                  (notebook 01)
    └── Database: DemoDB
        ├── User:   DemoUser → db_owner                 (notebook 01)
        └── Schema: Sales
            ├── Table:     Sales.Customer               (notebook 01 — 5 rows)
            ├── View:      Sales.vw_CustomerSummary     (notebook 01)
            ├── Table:     Sales.Product                (notebook 02 — 5 products)
            ├── Table:     Sales.CustomerOrder          (notebook 02 — 5 orders)
            ├── Table:     Sales.CustomerOrderLine      (notebook 02 — 8 seed lines)
            ├── Function:  Sales.fn_LineTotal           (notebook 02 — scalar)
            ├── Function:  Sales.fn_FullName            (notebook 02 — scalar)
            ├── Function:  Sales.fn_GetCustomerOrders   (notebook 02 — inline TVF)
            ├── Procedure: Sales.usp_SearchCustomers    (notebook 02)
            └── Procedure: Sales.usp_AddOrder           (notebook 02)
```

### Key concepts recap

| Concept | SQL Server | Snowflake equivalent |
|---------|-----------|---------------------|
| Local variable | `DECLARE @x TYPE; SET @x = val` | `LET x := val;` (Scripting) |
| Scalar function | `CREATE FUNCTION ... RETURNS type` | `CREATE FUNCTION ... RETURNS type` |
| Scalar UDF perf | Row-by-row — slow on large tables | Same risk for SQL UDFs |
| Inline TVF | `RETURNS TABLE AS RETURN (SELECT ...)` | UDTF with `TABLE(...)` |
| Parameterised TVF | `CROSS APPLY fn_Name(@param)` | `TABLE(fn_name(arg))` |
| Stored procedure | T-SQL — called with `EXEC` | JS/Python/Scripting — called with `CALL` |
| Default parameter | `@Param TYPE = default` | Same syntax |
| OUTPUT parameter | `@Param TYPE OUTPUT` + `SCOPE_IDENTITY()` | `RETURN value` |
| FK enforcement | Enforced by default | Metadata only — not enforced |

### What's next

**Notebook 03 — Querying and Data Shaping** uses everything built so far.
With a proper multi-table schema in place, the examples will be meaningful:
`INNER JOIN`, `LEFT JOIN`, `GROUP BY`, `HAVING`, subqueries, CTEs, and window functions.

### Optional Cleanup

> ⚠️ **Warning — Destructive and irreversible.**
> The cell below drops all objects created in **this notebook** in reverse dependency order.
> Objects from notebook 01 (`Sales.Customer`, `Sales.vw_CustomerSummary`, `DemoUser`, etc.)
> are **not** affected — use notebook 01's cleanup cell to remove those.
>
> **To run:** uncomment the SQL, then execute the cell.

In [ ]:
-- ─── Section 7: Cleanup — Uncomment to run ──────────────────────────────────
-- Drops only objects created in notebook 02, in reverse dependency order.
-- Objects from notebook 01 are NOT touched.

-- USE DemoDB;
-- GO

-- -- 1. Stored procedures
-- DROP PROCEDURE IF EXISTS [Sales].[usp_AddOrder];
-- DROP PROCEDURE IF EXISTS [Sales].[usp_SearchCustomers];
-- GO

-- -- 2. Functions — drop TVF before scalars it depends on
-- DROP FUNCTION IF EXISTS [Sales].[fn_GetCustomerOrders];
-- DROP FUNCTION IF EXISTS [Sales].[fn_LineTotal];
-- DROP FUNCTION IF EXISTS [Sales].[fn_FullName];
-- GO

-- -- 3. Tables — drop children before parents (FK constraints require this order)
-- DROP TABLE IF EXISTS [Sales].[CustomerOrderLine];
-- DROP TABLE IF EXISTS [Sales].[CustomerOrder];
-- DROP TABLE IF EXISTS [Sales].[Product];
-- GO